# pa2-judge — Task 7: pairwise judging with Kaggle model proxy
Must run as a **Kaggle Benchmarks task notebook** (so `kaggle_benchmarks` + the free model proxy are available): create it from kaggle.com/benchmarks (Create task) or Copy & Edit a benchmark starter notebook, then paste these cells. Accelerator: None. Internet on. **Add Input → Notebook Output → pa2-generate.**
Batches of 5, ≥5 s sleep before every request, 3 retries, judgments checkpointed after every batch, failures logged, `--reuse_judgments` resumes. Run the smoke cell interactively first, then **Save & Run All (Commit)**.

In [ ]:
GH_USER, REPO = "mohdddd", "small-model-alignment-starter"
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/{GH_USER}/{REPO}.git repo
%cd /kaggle/working/repo
!git log --oneline -1
import os, sys, json, shutil
sys.path.insert(0, "/kaggle/working/repo/src"); sys.path.insert(0, "/kaggle/working/repo/scripts")
from pa2.utils import find_one
import judge_pairwise_kaggle as J
os.makedirs("outputs/pairs", exist_ok=True); os.makedirs("outputs/eval", exist_ok=True)
PAIRS = ["uf_dpo_vs_sft_alpaca", "hh_dpo_vs_sft_hh", "lora_vs_base_persona"]
for name in PAIRS:
    shutil.copy(find_one(f"pairs/{name}.jsonl"), f"outputs/pairs/{name}.jsonl")
JUDGE = "google/gemini-3.6-flash"   # change if not available; record the one you use in the report

In [ ]:
import kaggle_benchmarks as kbench
try:
    print(sorted(kbench.llms.keys()))
except Exception as e:
    print("could not list models:", e, kbench.llms)

Smoke test: one batch of 5 pairs. Check that the output parses and A/B maps to model names.

In [ ]:
s = J.main(["--pairs_path", "outputs/pairs/uf_dpo_vs_sft_alpaca.jsonl", "--output_path", "outputs/eval/smoke_judged.jsonl",
            "--judge_model", JUDGE, "--limit", "5", "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3"])
!head -n 2 outputs/eval/smoke_judged.jsonl

Full judging (~125 requests, ~30–45 min). Safe to re-run: completed pairs are skipped.

In [ ]:
summaries = {}
for name in PAIRS:
    summaries[name] = J.main(["--pairs_path", f"outputs/pairs/{name}.jsonl", "--output_path", f"outputs/eval/{name}_judged.jsonl",
                              "--judge_model", JUDGE, "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3",
                              "--reuse_judgments"])

In [ ]:
for name, s in summaries.items():
    o = s["overall"]
    print(f"\n== {name}: {s['candidate']} vs {s['baseline']} | judge {s['judge_model']}")
    print(f"   judged {s['n_judged']}/{s['n_pairs']}  failed {s['n_failed_unjudged']}  "
          f"W/L/T/U = {o['wins']}/{o['losses']}/{o['ties']}/{o['unknown']}")
    print(f"   win rate excl ties {o['win_rate_excl_ties']}, ties as half {o['win_rate_ties_half']}")
    print("   position counts (bias check):", s["position_counts"])
    if "persona" in s:
        print("   persona criteria:", {k: (v["wins"], v["losses"], v["ties"]) for k, v in s["persona"].items()})
    for cat, c in s["by_category"].items():
        print(f"     {cat:28s} W/L/T {c['wins']}/{c['losses']}/{c['ties']}")

Verbosity check: candidate win rate when its answer is longer / similar / shorter than the baseline's.

In [ ]:
import analyze_judgments as A
for name in PAIRS:
    print(f"\n== {name}")
    A.main(["--pairs", f"outputs/pairs/{name}.jsonl", "--judged", f"outputs/eval/{name}_judged.jsonl",
            "--out", f"outputs/eval/{name}_length_analysis.json"])

In [ ]:
!cd outputs && zip -qr /kaggle/working/judge_outputs.zip eval pairs && ls -la eval